In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:


from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import tensorflow as tf

uploaded = files.upload()
file_name = next(iter(uploaded))

df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())


In [ ]:
# Cell 2: Data Cleaning

# Remove duplicate rows
df = df.drop_duplicates()

# Display missing values
print("Missing values before cleaning:")
print(df.isnull().sum())

# Check required columns
print("\nDataset columns:")
print(df.columns.tolist())

# Remove rows without a rating
df = df.dropna(subset=['rating'])

# Convert ratings to numeric values
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

# Remove invalid ratings
df = df.dropna(subset=['rating'])
df = df[df['rating'].between(1, 5)]

# Fill missing review text
df['review_content'] = df['review_content'].fillna('')

# Remove rows with empty review text
df = df[df['review_content'].str.strip() != '']

# Reset row numbers
df = df.reset_index(drop=True)

print("\nDataset shape after cleaning:", df.shape)
print("\nRating distribution:")
print(df['rating'].value_counts().sort_index())

display(df[['review_content', 'rating']].head())

In [ ]:
# Cell 3: Text Preprocessing and Rating Classes

import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Clean review text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_review'] = df['review_content'].apply(clean_text)

# Convert ratings to nearest whole-star classes
df['rating_class'] = df['rating'].round().astype(int)

print("Cleaned review examples:")
display(df[['review_content', 'clean_review', 'rating', 'rating_class']].head())

print("\nRating classes:")
print(df['rating_class'].value_counts().sort_index())

# Split the dataset into training and testing data
X = df['clean_review'].values
y = df['rating_class'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:

# Cell 4: Tokenization and Padding

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Maximum number of words the model will learn
MAX_WORDS = 10000

# Maximum review length
MAX_LEN = 150

# Create tokenizer
tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

# Learn vocabulary from training reviews only
tokenizer.fit_on_texts(X_train)

# Convert training and testing reviews into sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Make all reviews the same length
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("Training data shape:", X_train_pad.shape)
print("Testing data shape:", X_test_pad.shape)
print("Vocabulary size:", min(len(tokenizer.word_index) + 1, MAX_WORDS))

print("\nExample review:")
print(X_train[0])

print("\nNumerical sequence:")
print(X_train_pad[0][:30])


In [ ]:

# Cell 5: Build the BiLSTM Model

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    SpatialDropout1D,
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

# Set random seeds for reproducibility
tf.keras.utils.set_random_seed(42)

# Build model
model = Sequential([
    Embedding(
        input_dim=10000,
        output_dim=64,
        input_length=150
    ),

    SpatialDropout1D(0.3),

    Bidirectional(
        LSTM(32)
    ),

    Dense(32, activation='relu'),

    Dropout(0.4),

    Dense(5, activation='softmax')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
model.summary()


In [ ]:

# Cell 6: Train the Model

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Stop training when validation loss stops improving
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

# Reduce learning rate if validation loss stops improving
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=0.00001
)

# Train the model
history = model.fit(
    X_train_pad,
    y_train_encoded,
    epochs=20,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


In [ ]:

# Cell 7: Model Evaluation

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt

# Evaluate model
test_loss, test_accuracy = model.evaluate(
    X_test_pad,
    y_test_encoded,
    verbose=0
)

# Predict ratings
y_prob = model.predict(X_test_pad, verbose=0)
y_pred_encoded = np.argmax(y_prob, axis=1)

# Convert predictions back to ratings 1–5
y_pred = y_pred_encoded + 1
y_actual = y_test_encoded + 1

# Calculate evaluation metrics
accuracy = accuracy_score(y_actual, y_pred)
mae = mean_absolute_error(y_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))

print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Mean Absolute Error (MAE): {mae:.3f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")

print("\nClassification Report:")
print(classification_report(
    y_actual,
    y_pred,
    labels=[1, 2, 3, 4, 5],
    zero_division=0
))

# Confusion matrix
cm = confusion_matrix(y_actual, y_pred, labels=[1, 2, 3, 4, 5])

plt.figure(figsize=(7, 5))
plt.imshow(cm)
plt.colorbar()
plt.xticks([0, 1, 2, 3, 4], [1, 2, 3, 4, 5])
plt.yticks([0, 1, 2, 3, 4], [1, 2, 3, 4, 5])
plt.xlabel("Predicted Rating")
plt.ylabel("Actual Rating")
plt.title("Amazon Rating Prediction - Confusion Matrix")

for i in range(5):
    for j in range(5):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()


In [ ]:

# Cell 8: Predict Rating for a New Review

def predict_rating(review):
    # Clean the input review
    cleaned_review = clean_text(review)

    # Convert text into numerical sequence
    sequence = tokenizer.texts_to_sequences([cleaned_review])

    # Apply padding
    padded = pad_sequences(
        sequence,
        maxlen=MAX_LEN,
        padding='post',
        truncating='post'
    )

    # Predict probabilities
    probabilities = model.predict(padded, verbose=0)[0]

    # Get predicted rating (1 to 5 stars)
    predicted_rating = np.argmax(probabilities) + 1

    print("Customer Review:", review)
    print("Predicted Rating:", predicted_rating, "out of 5")
    print("Prediction Confidence:", f"{np.max(probabilities) * 100:.2f}%")

    return predicted_rating


# Test with a new review
predict_rating(
    "This product is excellent, good quality and worth the money"
)



In [ ]:

# Cell 9: Save Model and Tokenizer

import pickle
import os

# Create a folder for project files
os.makedirs("amazon_rating_model", exist_ok=True)

# Save the trained deep learning model
model.save("amazon_rating_model/amazon_rating_model.keras")

# Save the tokenizer
with open("amazon_rating_model/tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

print("Model and tokenizer saved successfully!")

print("Saved files:")
print(os.listdir("amazon_rating_model"))


In [ ]:

import shutil
from google.colab import files

shutil.make_archive(
    "amazon_rating_model",
    "zip",
    "amazon_rating_model"
)

files.download("amazon_rating_model.zip")
